# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema, accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install the `mlcroissant` library if not already installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Examine the available record sets, their `@id`s, fields, and columns.

All references to dataset components use their `@id` fields for consistency and reproducibility.

In [ ]:
from pprint import pprint

# List all record sets and their field/column IDs
record_sets = list(dataset.record_sets)
print(f"\nAvailable Record Sets ({len(record_sets)}):\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"    - Field: {getattr(field, 'name', '')} (@id: {field.id})")
        if hasattr(field, 'columns'):
            print("      Columns:")
            for col in field.columns:
                print(f"        - Column: {getattr(col, 'name', '')} (@id: {col.id})")
    print()

## 3. Data Extraction
Load one or more record sets into Pandas DataFrames for further exploration.

- Reference record sets and fields exclusively by their `@id` attributes.

In [ ]:
# Identify list of record set @id's from the overview above.
# For this dataset, let's extract all available record sets dynamically.
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for RecordSet @id: {record_set_id}")
            print("Columns:", dataframes[record_set_id].columns.tolist())
    except Exception as ex:
        print(f"Could not load data for RecordSet {record_set_id}: {ex}")

# Display the first few rows from (if present) the first DataFrame
if dataframes:
    primary_record_set_id = next(iter(dataframes))
    print(f"\nSample data from RecordSet @id: {primary_record_set_id}")
    display(dataframes[primary_record_set_id].head())
else:
    print("No records loaded from record sets.")

## 4. Exploratory Data Analysis (EDA)
Perform some example processing using field `@id`s!

- Filter records by a numeric field
- Normalize the field
- Group/analyze as a demonstration

Replace the example field IDs below with those found in the data overview if needed.

In [ ]:
# --- Set the record set and field IDs here ---

if dataframes:
    record_set_id = primary_record_set_id
    df = dataframes[record_set_id]
    print(f"EDA for RecordSet: {record_set_id}")

    # Guess a numeric field by attempting to convert columns
    numeric_field_id = None
    for col in df.columns:
        # Try to convert the column to float, if possible
        try:
            numeric_data = pd.to_numeric(df[col], errors='coerce')
            if numeric_data.notnull().sum() > 0:
                numeric_field_id = col
                df[col] = numeric_data
                break
        except Exception:
            continue

    if not numeric_field_id:
        print("No numeric field found for EDA.")
    else:
        print(f"Numeric field used for EDA: {numeric_field_id}")
        # Example threshold: mean value
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Demonstrate grouping if a likely categorical column exists
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() > 1 and df[col].nunique() < len(df) / 2:
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field for aggregation found.")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Visualize the numeric field's distribution and (if available) its relationship with a group field.

In [ ]:
if dataframes and numeric_field_id:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=30, edgecolor='k')
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8, 4))
        df.groupby(group_field_id)[numeric_field_id].mean().plot(kind='bar')
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to:

- Load metadata and record sets from a Croissant dataset using its schema URL
- Overview available record sets and their contents using `@id` references
- Load and inspect record sets with `mlcroissant`, showing field and column names
- Apply simple EDA steps including numeric filtering, normalization, and grouping
- Visualize data distributions and relationships

**Further steps:**
- Integrate domain-specific preprocessing based on the actual field meanings (consult field `@id`s and documentation)
- Explore more in-depth visualizations and statistical tests
- Join or link record sets using shared fields and their `@id`s

For more, visit the [mlcroissant documentation](https://mlcommons.github.io/croissant/api.html).